In [1]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
import time
from selenium.webdriver.common.action_chains import ActionChains
import pandas as pd
import datetime as dt
import os

In [8]:
import sys
from pathlib import Path
sys.path.append('../../src')

# Agora o import funciona começando por 'src'
from extractor_ranking import main as exr

In [ ]:
def get_navegacao(driver):
    div_paginador = driver.find_element(By.XPATH, '//div[@id="form:tabView:dataTable0_paginator_top"]')
    
    numero_insercoes = int(div_paginador.find_element(By.XPATH, './/span[@class="ui-paginator-current"]').text.split()[0].replace('.',''))
    numero_paginas = numero_insercoes//50+1
    pagina_corrente = int(div_paginador.find_element(By.XPATH, './/a[contains(@class,"ui-state-active")]').text)
    
    return numero_insercoes, pagina_corrente, numero_paginas
    
def bt_navegacao(driver, tp_navegacao):
    div_paginador = driver.find_element(By.XPATH, '//div[@id="form:tabView:dataTable0_paginator_top"]')
    
    bt = div_paginador.find_element(By.XPATH, './/a[contains(@class, "ui-paginator-' + tp_navegacao + '")]')
    
    try:
        bt.click()
        print('Navegação do tipo: ' + tp_navegacao)
    except:
        print('Navegação não foi possível: ' + tp_navegacao)

    # bt_first = div_paginador.find_element(By.XPATH, './/a[contains(@class, "ui-paginator-first")]')
    # bt_prev = div_paginador.find_element(By.XPATH, './/a[contains(@class, "ui-paginator-prev")]')
    # bt_next = div_paginador.find_element(By.XPATH, './/a[contains(@class, "ui-paginator-next")]')
    # bt_last = div_paginador.find_element(By.XPATH, './/a[contains(@class, "ui-paginator-last")]')

def get_col_names(elemento_th):
    nome_colunas = []
    for coluna in elemento_th:
        # print(coluna)
        texto = coluna.text.replace('.','')
        # print(texto)
        
        if texto == '#':
            texto = 'posicao'

        if texto != '':
            nome_colunas.append(texto)    

    # trata colunas com nomes repetido
    for coluna in nome_colunas:
        indices_repetido = [i for i, c in enumerate(nome_colunas) if c == coluna]
        if len(indices_repetido) > 1:
            for i in range(len(indices_repetido)):
                nome_colunas[indices_repetido[i]] = nome_colunas[indices_repetido[i]] + '_' + str(i + 1)
            
    # len(nome_colunas) 
    return nome_colunas

def get_ranking_page(driver):
    # thead_usuario = driver.find_elements(By.XPATH, '//thead[@id="form:dataUser_head"]')
    # tbody_usuario = driver.find_elements(By.XPATH, '//tbody[@id="form:dataUser_data"]')
    thead_ranking = driver.find_elements(By.XPATH, '//thead[@id="form:tabView:dataTable0_head"]')
    tbody_ranking = driver.find_elements(By.XPATH, '//tbody[@id="form:tabView:dataTable0_data"]')
    
    elemento_hbody = thead_ranking[0]
    elemento_tbody = tbody_ranking[0]
    
    # Seleciona linhas
    nome_colunas = get_col_names(elemento_hbody.find_elements(By.TAG_NAME, "th"))
    linhas = elemento_tbody.find_elements(By.XPATH, './/tr[@data-ri]')
    print('Quantidade de linhas:', len(linhas),end='')
    
    lista_pontos = []
    lista_acertos = []
    lista_erros = []
    lista_brancos = []
    lista_percentual = []
    
    for linha in linhas:
        # Pega todas as células da linha
        celulas = linha.find_elements(By.TAG_NAME, "td")
        
        if not celulas:
            # Pula se for a linha do cabeçalho (que pode ter <th>)
            continue
            
        registro = {
            'posicao':celulas[0].text,
            'usuario':celulas[1].text,
        }
    
        registro_pontos = registro.copy()
        registro_acertos = registro.copy()
        registro_erros = registro.copy()
        registro_brancos = registro.copy()
        registro_percentual = registro.copy()
    
        n_cols = len(nome_colunas) - 2
        # print(n_col)
        
        for n_col in range(n_cols):
            nome_coluna = nome_colunas[n_col + 2]
            # print(nome_coluna)
            
            # texto = celulas[n_col+2].text
            # print(texto)
            # raise SairDoLoop()texto.split()
            
            lista_valores = celulas[n_col+2].text.replace('|','').split()
    
            # print(len(lista_valores))
            # Valores normais de disciplina sem brancos
            if len(lista_valores) >= 4:
                # print(lista_valores)
                pontos, acertos, erros  = lista_valores[0:3]
                percentual = lista_valores[-1]
                
            # Valores normais de disciplina com brancos
            if len(lista_valores) > 4:
                brancos = lista_valores[-2]
            else:
                brancos = 0
                
            # Valores normais de disciplina com brancos
            if len(lista_valores) == 0:
                pontos  = 0
                percentual = '0%'
            elif len(lista_valores) < 4:
                pontos  = lista_valores[0]
                percentual = lista_valores[-1]
    
            registro_pontos[nome_coluna] = pontos
            registro_percentual[nome_coluna] = percentual
    
            if len(lista_valores) >= 4:
                registro_acertos[nome_coluna] = acertos
                registro_erros[nome_coluna] = erros
                registro_brancos[nome_coluna] = brancos
            
            # informações de titulos
            # if len(lista_valores) < 4:
            #     print(registro_pontos)
            #     raise SairDoLoop()
        
        lista_pontos.append(registro_pontos)
        lista_acertos.append(registro_acertos)
        lista_erros.append(registro_erros)
        lista_brancos.append(registro_brancos)
        lista_percentual.append(registro_percentual)
    
        print('.',end='')
    
    return lista_pontos, lista_acertos, lista_erros, lista_brancos, lista_percentual

In [10]:
# Inicializa navegador "invisível"
options = uc.ChromeOptions()
# options.add_argument("--headless")  # opcional, sem janela visível
driver = uc.Chrome(options=options, version_main=145)

# Acessa a página protegida
driver.get("https://olhonavaga.com.br/login")

# # Aguarda Cloudflare (e CAPTCHA, se possível)
# time.sleep(10)  # ou use WebDriverWait para detectar o fim da verificação


while True:
    page_state = driver.execute_script('return document.readyState;')
    print("Estado da página:", page_state)
    if page_state == 'complete':
        break
    time.sleep(1)

Estado da página: complete


In [12]:
# Preenche o formulário (se passou pelo desafio)
driver.find_element(By.NAME, "form:email").send_keys("scarlosfreitas@gmail.com")
driver.find_element(By.NAME, "form:password").send_keys("gv8b3t22yh8r")

driver.find_element(By.NAME, "form:signInButton").click()

# Espera resposta
while True:
    page_state = driver.execute_script('return document.readyState;')
    print("Estado da página:", page_state)
    if page_state == 'complete':
        break
    time.sleep(1)

Estado da página: complete


In [14]:
dict_fisco = {
    'sefaz-pi':{'link':"https://olhonavaga.com.br/rankings/ranking?id=80578"},
    'sefaz-pr':{'link':"https://olhonavaga.com.br/rankings/ranking?id=79110"},
    'sefaz-pe':{'link':"https://olhonavaga.com.br/rankings/ranking?id=49551"},
    'sefaz-mg':{'link':"https://olhonavaga.com.br/rankings/ranking?id=47050"},
    'sefaz-mt':{'link':"https://olhonavaga.com.br/rankings/ranking?id=52584"},
}

fisco_escolhido = 'sefaz-pi'
driver.get(dict_fisco[fisco_escolhido]['link'])

# Espera resposta
while True:
    page_state = driver.execute_script('return document.readyState;')
    print("Estado da página:", page_state)
    if page_state == 'complete':
        break
    time.sleep(1)

Estado da página: complete


In [15]:
numero_insercoes, pagina_corrente, numero_paginas = exr.get_navegacao(driver)

print('Fisco: {} Inserções: {} Pagina: {}/{}'.format(fisco_escolhido.upper(), numero_insercoes,pagina_corrente,numero_paginas))
# elemento[0].get_attribute('outerHTML') 

if pagina_corrente != 1:
    bt_navegacao(driver, 'first')
    time.sleep(10)
    
lista_pontos, lista_acertos, lista_erros, lista_brancos, lista_percentual = [],[],[],[],[]

Fisco: SEFAZ-PI Inserções: 569 Pagina: 1/12


In [18]:
for i in range(1,numero_paginas+1):
    _, pagina_corrente, _ = exr.get_navegacao(driver)

    print('Pagina {}/{}'.format(pagina_corrente,numero_paginas))

    l_pontos, l_acertos, l_erros, l_brancos, l_percentual = exr.get_ranking_page(driver)

    lista_pontos = lista_pontos + l_pontos
    lista_acertos = lista_acertos + l_acertos
    lista_erros = lista_erros + l_erros
    lista_brancos = lista_brancos + l_brancos
    lista_percentual = lista_percentual + l_percentual

    print('')
          
    if pagina_corrente != numero_paginas:
        exr.bt_navegacao(driver, 'next')
        time.sleep(10)
    else:
        print('Fim do loop')
        break

Pagina 1/12
Quantidade de linhas: 50..................................................
Navegação do tipo: next
Pagina 2/12
Quantidade de linhas: 50..................................................
Navegação do tipo: next
Pagina 3/12
Quantidade de linhas: 50..................................................
Navegação do tipo: next
Pagina 4/12
Quantidade de linhas: 50..................................................
Navegação do tipo: next
Pagina 5/12
Quantidade de linhas: 50..................................................
Navegação do tipo: next
Pagina 6/12
Quantidade de linhas: 50..................................................
Navegação do tipo: next
Pagina 7/12
Quantidade de linhas: 50..................................................
Navegação do tipo: next
Pagina 8/12
Quantidade de linhas: 50..................................................
Navegação do tipo: next
Pagina 9/12
Quantidade de linhas: 50..................................................
Navegação do tipo: next
P

In [26]:
df_pontos = pd.DataFrame(lista_pontos, dtype='string')
df_acertos = pd.DataFrame(lista_acertos, dtype='string')
df_erros = pd.DataFrame(lista_erros, dtype='string')
df_brancos = pd.DataFrame(lista_brancos, dtype='string')
df_percentual = pd.DataFrame(lista_percentual, dtype='string')

In [27]:
# Constroi pasta destino
string_data = dt.datetime.now().strftime("%Y-%m-%d")
string_hora = dt.datetime.now().strftime("%H-%M")

pasta_destino = './' + fisco_escolhido + '/' + string_data
os.makedirs(pasta_destino, exist_ok=True)

In [28]:
print('Pasta destino: {}'.format(pasta_destino))

Pasta destino: ./sefaz-pi/2025-09-20


In [29]:
df_pontos.to_parquet(pasta_destino + '/df_pontos_' + string_data + '_' + string_hora + '.parquet', compression='snappy')
df_acertos.to_parquet(pasta_destino + '/df_acertos_' + string_data + '_' + string_hora + '.parquet', compression='snappy')
df_erros.to_parquet(pasta_destino + '/df_erros_' + string_data + '_' + string_hora + '.parquet', compression='snappy')
df_brancos.to_parquet(pasta_destino + '/df_brancos_' + string_data + '_' + string_hora + '.parquet', compression='snappy')
df_percentual.to_parquet(pasta_destino + '/df_percentual_' + string_data + '_' + string_hora + '.parquet', compression='snappy')
    